In [204]:
import openpyxl
from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
import os

filename = "rack_layout_top_down.xlsx"
if os.path.exists(filename):
    os.remove(filename)
    print(f"Existing file '{filename}' deleted.")

def generate_visual(distribution, loc):
    
    node_positions = distribution['stable_placement']
    try:
        wb = openpyxl.load_workbook(filename)
    except FileNotFoundError:
        wb = openpyxl.Workbook()
        if 'Sheet' in wb.sheetnames:
            del wb['Sheet']
    
    tab_name = f"Rack_{loc}"
    if tab_name in wb.sheetnames:
        del wb[tab_name]
    ws = wb.create_sheet(title=tab_name)

    # Set column widths
    ws.column_dimensions['A'].width = 25
    ws.column_dimensions['B'].width = 25
    ws.column_dimensions['C'].width = 15
    ws.column_dimensions['D'].width = 15

    # Define styles
    header_font = Font(bold=True)
    border = Border(left=Side(style='thin'), right=Side(style='thin'), 
                   top=Side(style='thin'), bottom=Side(style='thin'))
    center_aligned = Alignment(horizontal='center', vertical='center')
    total_font = Font(bold=True)
    total_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')

    color_map = {
        'compute_nodes': 'FF9999',
        'GPU_nodes': '99CCFF',
        'LAN': '99FF99',
    }

    # Write headers
    headers = ["Rack Position", "Device", "Wattage (W)", "Weight (kg)"]
    for col, header in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.font = header_font
        cell.border = border
        cell.alignment = center_aligned

    # Track occupied positions
    occupied_info = {}
    total_wattage = 0
    total_weight = 0
    occupied_positions = {}
    occupied_label = {}

    # First pass: Identify all positions and their heights
    for device, posy in node_positions.items():
        if 'LAN_' in device:
            lan_parts = '_'.join(device.split('_')[:-1]).split('__')
            lan_type = lan_parts[0]
            switch_model = lan_parts[1]
            for switch in LANs[lan_type]['switch']:
                if switch['model'] == switch_model:
                    height = switch['height']
                    wattage = switch['wattage']
                    weight = switch['weight']
                    break
            
            pos = posy + height - 1
            occupied_label[pos] = f"{pos-height+1}-{pos}"
            for i in range(pos-height+1, pos+1):
                occupied_positions[i] = device
            
            occupied_info[pos] = {
                'device': device,
                'type': 'LAN Switch',
                'height': height,
                'color': color_map.get('LAN', 'FFFFFF'),
                'wattage': wattage,
                'weight': weight
            }
            total_wattage += wattage
            total_weight += weight
        else:
            node_type = '_'.join(device.split('_')[:-1])
            height = colors_info[node_type]['height']
            wattage = colors_info[node_type]['wattage']
            weight = colors_info[node_type]['weight']
            pos = posy + height - 1
            occupied_label[pos] = f"{pos-height+1}-{pos}"
            for i in range(pos-height+1, pos+1):
                occupied_positions[i] = device
            
            occupied_info[pos] = {
                'device': device,
                'type': node_type.replace('_', ' ').title(),
                'height': height,
                'color': color_map.get(node_type, 'FFFFFF'),
                'wattage': wattage,
                'weight': weight
            }
            total_wattage += wattage
            total_weight += weight

    # Second pass: Write data to worksheet
    current_row = 2
    for rack_pos in range(42, 0,-1):
        # Skip if this position is covered by a device that starts above
        if rack_pos in occupied_positions and rack_pos not in occupied_info:
            continue
            
        # Get position label
        position_label = occupied_label.get(rack_pos, str(rack_pos))
        
        # Write rack position
        ws.cell(row=current_row, column=1, value=position_label)
        ws.cell(row=current_row, column=1).border = border
        ws.cell(row=current_row, column=1).alignment = center_aligned

        if rack_pos in occupied_info:
            device_info = occupied_info[rack_pos]
            height = device_info['height']
            
            # Write device info FIRST
            ws.cell(row=current_row, column=2, value=device_info['device'])
            ws.cell(row=current_row, column=3, value=device_info['wattage'])
            ws.cell(row=current_row, column=4, value=device_info['weight'])
            
            # Apply styling to all cells that will be merged
            fill = PatternFill(start_color=device_info['color'], end_color=device_info['color'], fill_type='solid')
            for col in range(1, 5):
                ws.cell(row=current_row, column=col).fill = fill
                ws.cell(row=current_row, column=col).border = border
                ws.cell(row=current_row, column=col).alignment = center_aligned
            
            # Merge cells if height > 1
            if height > 1:
                for col in range(1, 5):
                    ws.merge_cells(
                        start_row=current_row,
                        end_row=current_row + height - 1,
                        start_column=col,
                        end_column=col
                    )
                
                # Apply border to all merged cells
                for row in range(current_row, current_row + height):
                    for col in range(1, 5):
                        ws.cell(row=row, column=col).border = border
            
            current_row += height
        else:
            # Empty position
            for col in range(2, 5):
                ws.cell(row=current_row, column=col, value="")
                ws.cell(row=current_row, column=col).border = border
                ws.cell(row=current_row, column=col).alignment = center_aligned
            
            current_row += 1

    # Add total row
    total_row = current_row + 1
    ws.cell(row=total_row, column=1, value="TOTAL").font = total_font
    ws.cell(row=total_row, column=2, value="").font = total_font
    ws.cell(row=total_row, column=3, value=total_wattage).font = total_font
    ws.cell(row=total_row, column=4, value=total_weight).font = total_font
    
    for col in range(1, 5):
        ws.cell(row=total_row, column=col).border = border
        ws.cell(row=total_row, column=col).fill = total_fill
        ws.cell(row=total_row, column=col).alignment = center_aligned

    # Save the workbook
    wb.save(filename)
    print(f"Excel file '{filename}' has been created with positions from 42 at top to 1 at bottom.")

Existing file 'rack_layout_top_down.xlsx' deleted.


In [205]:
def iterate_on_racks():
    for i in distributions_info:
        if distributions_info[i].get('stable_placement',0) != 0:
            generate_visual(distributions_info[i],i)
    return

In [206]:
import pickle
from collections import Counter
from math import floor
def global_main():
    global colors_info, LANs, Cables, max_box_wattage, max_box_height,  rack_height_u , rack_weight_kg, rack_width_mm, \
            rack_depth_mm, u_height_mm, rack_height_mm, rack_height_u, rack_cg_height_mm, unit_to_cm, u_height_mm, \
            device_to_rackside, racktop_to_ceiling, rack_to_rack, Rack_rows, Rack_rows, distributions_info
    


    with open('Latest_Racks_info.pkl', 'rb') as f:
        colors_info, LANs, Cables, Rack_rows, distributions_info = pickle.load(f)
    
    
    max_box_wattage = 16000
    max_box_height = rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 750
    rack_depth_mm = 1200
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2
    global_rack_signature = dict()
    unit_to_cm = u_height_mm /10
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cm
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    # adding to the Cables the maximum stretch
    for switch in Cables:
        for cable in Cables[switch]:
            cable['in_rack_stretch'] = floor((cable['length']*100/unit_to_cm) - (2* device_to_rackside))
            
            
if __name__ == "__main__":
    global_main()
    iterate_on_racks()

Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.


In [207]:
distributions_info[0]['stable_placement']

{'compute_nodes_1': 33,
 'LAN_3__S_xx64_2': 21,
 'LAN_2__Z_9xxx_3': 19,
 'GPU_nodes_4': 15,
 'compute_nodes_5': 13}

In [208]:
colors_info

{'compute_nodes': {'count': 2,
  'wattage': 1350,
  'height': 2,
  'weight': 28.7,
  'LAN_1': {'count': 0, 'speed': 400},
  'LAN_2': {'count': 1, 'speed': 200},
  'LAN_3': {'count': 1, 'speed': 25},
  'LAN_4': {'count': 0, 'speed': 25},
  'LAN_5': {'count': 0, 'speed': 1},
  'LAN_6': {'count': 0, 'speed': 400},
  'type': 'compute_nodes'},
 'GPU_nodes': {'count': 1,
  'wattage': 11660,
  'height': 4,
  'weight': 61.4,
  'LAN_1': {'count': 0, 'speed': 400},
  'LAN_2': {'count': 4, 'speed': 400},
  'LAN_3': {'count': 1, 'speed': 25},
  'LAN_4': {'count': 0, 'speed': 25},
  'LAN_5': {'count': 0, 'speed': 1},
  'LAN_6': {'count': 0, 'speed': 400},
  'type': 'GPU_nodes'},
 'NVMenodes': {'count': 1,
  'wattage': 1128,
  'height': 1,
  'weight': 20,
  'LAN_1': {'count': 2, 'speed': 400},
  'LAN_2': {'count': 2, 'speed': 200},
  'LAN_3': {'count': 1, 'speed': 25},
  'LAN_4': {'count': 0, 'speed': 25},
  'LAN_5': {'count': 0, 'speed': 1},
  'LAN_6': {'count': 0, 'speed': 400},
  'type': 'NVMenod